# LAB 3 - Classical MD
**Simulating CLN025 with GROMACS**


Authors:

- Prof. Marco A. Deriu (marco.deriu@polito.it)
- Eric A. Zizzi (eric.zizzi@polito.it)
- Marcello Miceli (marcello.miceli@polito.it)

# Table of Contents

1. What a simulation needs
2. Building the system
3. Energy minimisation
4. Equilibration
5. Production at three temperatures
6. A second force field
7. Running the simulations
8. A first look at the results
9. Exercises
10. Your deliverable

The analysis of the trajectories continues in LAB 4.

**Learning outcomes:**
- build a solvated, neutral simulation box for a protein with GROMACS
- choose the shape and size of the box, and defend the choice
- run energy minimisation, NVT and NPT equilibration and production, and check each stage
- explain what each setting of a parameter file does
- run long simulations in the background and measure their speed
- set up the same molecule with a second force field

# Before you start

You need three things from the previous labs:

- **your ns/day** from LAB 0: it tells you how long each simulation takes on your computer;
- **setup.sh** from LAB 1: it creates the working folder that this lab fills (a solution is in `../01-LinuxBash/solution_setup.sh`);
- **VMD** from LAB 2, to look at what you build.

Everything happens in this folder, `~/molbiomech/labs/03-ClassicalMD`. The short steps run in the notebook cells; the long production runs go in the background from a terminal, as you learned in LAB 1.

The molecule is the one of LAB 2: CLN025 from NMR model 1 of PDB 2RVD, with an acetyl cap (ACE) on Tyr1 and an amide cap (NH2) on Tyr10. Keep in mind that experimental values such as the melting temperature were measured on CLN025 **without** caps.

# 1. What a simulation needs

[GROMACS](https://www.gromacs.org) is a free, open-source and very fast program for molecular dynamics (MD). It is a set of command-line tools, all called as `gmx <tool>`; each one prints its help with `-h`:

In [ ]:
!gmx pdb2gmx -h 2>&1 | sed -n '/^SYNOPSIS/,/^OPTIONS/p' | head -n 30

## Three input files, one run file

| File | Content | Answers |
|---|---|---|
| Coordinates: `.gro` or `.pdb` | The position of every atom (and in `.gro` also velocities), and the box | **What** system? |
| Topology: `.top` and `.itp` | Every atom with its type, charge and mass; bonds, angles, dihedrals; the force-field parameters | **Which** model? |
| Parameters: `.mdp` | Algorithm, time step, length, cut-offs, temperature, pressure, output | **How** to simulate? |

`gmx grompp` (the GROMACS pre-processor) checks that the three agree and packs them into one binary **run input file**, `.tpr`. `gmx mdrun` runs it. Minimisation, equilibration and production are all this same pair of commands with a different `.mdp` file.

`.gro` and `.pdb` files both hold coordinates. `.gro` files use nm (`.pdb` files use Å), can hold velocities, and end with a line of box vectors.

<img src="imgs/MD_FlowChart.png" width="420">

The chart is also in `materials/MD_FlowChart.pdf`, and the theory behind it in `materials/Practical_MD.pdf`. In this lab each stage has its own numbered folder: `00-build`, `01-em`, `02-nvt`, `03-npt`, `04-md`.

<div class="alert alert-block alert-warning">
<b>GROMACS is a command-line program.</b> Unless you give a full path, it reads input files from the folder you call it from and writes its output there. If a file is missing, it stops with an error message. GROMACS error messages are explicit: <b>read the error</b>, and most problems solve themselves.
<br><br>
In this notebook each command sends what it prints to a <code>.out</code> file in its folder, and shows it only if the command fails (<code>|| tail -n 20 file.out</code>). Open the <code>.out</code> files to read everything GROMACS said.
</div>

# 2. Building the system

## 2.1 The working folder

Your `setup.sh` from LAB 1 creates the folder of one NMR model, copies the course parameter files into it and caps the model. Run it for model 1, the course's reference model (use your own script if it works):

In [ ]:
%%bash
bash ../01-LinuxBash/solution_setup.sh 1
ls cln025_model1 cln025_model1/00-build

## 2.2 The topology: `gmx pdb2gmx`

`pdb2gmx` reads the structure, matches every residue to a building block of the force field, adds the hydrogens, and writes three files: the coordinates (`.gro`), the topology (`.top`) and the position restraints (`.itp`, used in section 4).

In [ ]:
%%bash
cd cln025_model1/00-build
gmx pdb2gmx -f cln025_capped.pdb -o cln025.gro -p topol.top -i posre.itp \
            -ff amber99sb-ildn -water tip3p -ignh > pdb2gmx.out 2>&1 || tail -n 20 pdb2gmx.out
grep -E "Now there are|Total charge" pdb2gmx.out

| Option | Meaning |
|---|---|
| `-ff amber99sb-ildn` | The **force field**: where every parameter of the topology comes from. The course's main force field; section 6 adds a second one |
| `-water tip3p` | The water model that goes with AMBER force fields |
| `-ignh` | Ignore the hydrogens of the input file and rebuild them with the force field's names |
| `-i posre.itp` | Write the position-restraint file |

The caps ACE and NH2 are building blocks of the force field, so the chain has no charged ends. The total charge, −2, comes from Asp3 and Glu5.

<div class="alert alert-block alert-info">
Older versions of this course also passed <code>-heavyh</code>, which makes hydrogen atoms heavier so that a longer time step can be used. We keep real masses and a 2 fs time step, and fix the length of the bonds to hydrogen instead (<code>constraints = h-bonds</code> in the <code>.mdp</code> files).
</div>

## 2.3 Inside the topology

The topology is a text file made of sections in square brackets:

In [ ]:
!grep -n -E "^\[|^#" cln025_model1/00-build/topol.top

| Section | Content |
|---|---|
| `#include "amber99sb-ildn.ff/forcefield.itp"` | The force field: atom types with their Lennard-Jones parameters, and the bonded parameters |
| `[ moleculetype ]` | The name of the molecule |
| `[ atoms ]` | Every atom: type, residue, name, charge, mass |
| `[ bonds ]`, `[ angles ]`, `[ dihedrals ]` | Which atoms are bonded; the parameters come from the force field |
| `[ pairs ]` | Atoms three bonds apart (1–4 pairs), whose non-bonded interaction is scaled down |
| `#ifdef POSRES` | Position restraints, switched on only when asked (section 4) |
| `#include ".../tip3p.itp"`, `".../ions.itp"` | Water and ions |
| `[ system ]`, `[ molecules ]` | The name of the system, and how many copies of each molecule it holds, in order |

The first residues of `[ atoms ]`: every atom carries a partial charge; the charges of a residue add up to an integer.

In [ ]:
!sed -n '/\[ atoms \]/,/residue   2/p' cln025_model1/00-build/topol.top

The force field's `[ defaults ]` say how the non-bonded interactions are computed. `fudgeLJ` and `fudgeQQ` scale the Lennard-Jones and electrostatic interactions of the 1–4 pairs (by 0.5 and 0.8333 in AMBER), and `gen-pairs = yes` builds their parameters from the atom types:

In [ ]:
%%bash
top=$(gmx --version 2>&1 | grep "Data prefix" | awk '{print $3}')/share/gromacs/top
cat $top/amber99sb-ildn.ff/forcefield.itp
echo "--- TIP3P water:"
grep -v "^;" $top/amber99sb-ildn.ff/tip3p.itp | head -n 30

Water is one more molecule type, `SOL`: an oxygen and two hydrogens with fixed charges. Both hydrogens have the same atom type (same parameters) but different atom names. TIP3P water is rigid: `SETTLE` keeps its geometry fixed during the simulation, and the bonds and angle listed are used only when rigidity is switched off with `-DFLEXIBLE`.

The end of the topology lists the molecules of the system, in the same order as in the coordinate file. For now there is one peptide:

In [ ]:
!tail -n 12 cln025_model1/00-build/topol.top

## 2.4 The box: shape and size

MD simulations use **periodic boundary conditions**: the box is repeated in every direction, and a molecule that leaves through one face comes back through the opposite one. There are no walls, and no surface. Two rules follow:

1. **The peptide must not feel its own copies.** Interactions are computed up to a cut-off distance (1.0 nm in the course's AMBER settings), so the peptide and its periodic images must stay farther apart than that.
2. **Every atom costs time.** Almost all of the box is water, and the cost of a simulation grows with the number of atoms.

`gmx editconf` builds the box:

```bash
gmx editconf -f cln025.gro -o box.gro -c -d 1.0 -bt dodecahedron
```

It centres the peptide (`-c`) and builds a box of the chosen type (`-bt`) with at least `-d` nm between the peptide and the box faces. Two images of the peptide are then at least 2 × `d` apart.

| Option | Type | Description |
|---|---|---|
| `-f` | `.gro`, `.pdb` | Input structure |
| `-o` | `.gro`, `.pdb` | Output structure |
| `-n` | `.ndx` | Index file, to work on a subset of the atoms |
| `-bt` | name | Box type: `triclinic`, `cubic`, `dodecahedron`, `octahedron` |
| `-d` | number | Distance between the solute and the box faces (nm) |
| `-box` | vector | Box vector lengths, e.g. `-box 5 5 8` |
| `-c` | – | Centre the molecule in the box |
| `-princ` | – | Align the molecule along its principal axes |
| `-translate`, `-rotate` | vector | Move or rotate the coordinates |

### Which box?

Compare three box types and four distances. The next cell builds each box and fills it with water (section 2.5 explains `solvate`), and records the volume, the number of water molecules and the total number of atoms. It works in its own folder, `boxes`, so the real system is not touched.

In [ ]:
%%bash
mkdir -p boxes && cd boxes
cp ../cln025_model1/00-build/cln025.gro ../cln025_model1/00-build/topol.top .
for bt in cubic octahedron dodecahedron; do
    for d in 0.8 1.0 1.2 1.5; do
        cp topol.top topol_$bt$d.top
        gmx editconf -f cln025.gro -o box_$bt$d.gro -c -d $d -bt $bt > editconf.out 2>&1
        gmx solvate -cp box_$bt$d.gro -cs spc216.gro -o solvated_$bt$d.gro -p topol_$bt$d.top > solvate.out 2>&1
        volume=$(grep "new box volume" editconf.out | awk '{print $5}')
        waters=$(grep "^SOL" topol_$bt$d.top | awk '{print $2}')
        atoms=$(sed -n 2p solvated_$bt$d.gro)
        echo "$bt $d $volume $waters $atoms"
    done
done > boxes.txt
cat boxes.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rows = [line.split() for line in open("boxes/boxes.txt")]
shapes = ["cubic", "octahedron", "dodecahedron"]
table = {(r[0], float(r[1])): (float(r[2]), int(r[3]), int(r[4])) for r in rows}
distances = sorted({float(r[1]) for r in rows})

print(f"{'box':<13} {'d (nm)':>6} {'volume (nm3)':>12} {'waters':>7} {'atoms':>6} {'volume vs cube':>15}")
for shape in shapes:
    for d in distances:
        volume, waters, atoms = table[(shape, d)]
        print(f"{shape:<13} {d:6.1f} {volume:12.1f} {waters:7d} {atoms:6d} {volume / table[('cubic', d)][0]:15.2f}")

for shape in shapes:
    plt.plot(distances, [table[(shape, d)][2] for d in distances], "o-", label=shape)
plt.xlabel("distance to the box faces, d (nm)")
plt.ylabel("atoms in the box")
plt.legend()
plt.tight_layout()

At the same `d`, the peptide is equally far from its images in all three boxes, but the rhombic dodecahedron needs about 71% of the volume of the cube, and the truncated octahedron about 77%. Both are closer to a sphere, so less water sits in corners far from the peptide.

<div class="alert alert-block alert-info"><b>Choose and defend:</b> which box type and which <code>d</code> would you use, and why? What would you gain with <code>d</code> = 0.8 nm, and what would you risk?</div>

The course uses a **rhombic dodecahedron with d = 1.0 nm**: the peptide's images are at least 2.0 nm apart, twice the 1.0 nm cut-off, for 3,646 atoms before adding ions. That is safe for the folded hairpin. An unfolded chain is longer, and at 600 K (section 5) it may come close to its own images: in LAB 4 you will check it with `gmx mindist -pi`.

<div class="alert alert-block alert-info">Which terms of the potential energy act between every pair of atoms, and how do cut-offs and PME keep their cost from growing with the square of the number of atoms?</div>

In [ ]:
%%bash
cd cln025_model1/00-build
gmx editconf -f cln025.gro -o box.gro -c -d 1.0 -bt dodecahedron > editconf.out 2>&1 || tail -n 20 editconf.out
tail -n 1 box.gro

The last line of a `.gro` file holds the box vectors in nm. A cubic box needs three numbers; a dodecahedron is a triclinic box, described by nine.

To see it, open `box.gro` in VMD and type `pbc box` in the **Tk Console** (Extensions → Tk Console).

## 2.5 Water

`gmx solvate` fills the box with water, skipping positions that clash with the peptide, and adds the number of water molecules to the topology:

In [ ]:
%%bash
cd cln025_model1/00-build
gmx solvate -cp box.gro -cs spc216.gro -o solvated.gro -p topol.top > solvate.out 2>&1 || tail -n 20 solvate.out
tail -n 3 topol.top

`spc216.gro` is a small, pre-equilibrated box of 216 three-site water molecules that `solvate` repeats to fill any box. It suits every three-site model, TIP3P included: it provides only the starting positions, while the topology decides which water model is simulated.

<div class="alert alert-block alert-danger"><b>Run it once.</b> GROMACS does exactly what you ask. Running <code>solvate</code> twice fills the empty box twice and adds the water to the topology twice: the topology then lists twice as many water molecules as the coordinate file holds, and the next step stops with a <b>fatal error</b>. GROMACS keeps the previous topology as <code>#topol.top.1#</code>; if it happens, restore that file or rebuild from <code>pdb2gmx</code>.</div>

## 2.6 Ions

The peptide carries a charge of −2. Ions make the system neutral and bring the salt to a physiological 0.15 M. The tool, `gmx genion`, replaces water molecules with ions, and it reads a **run input file** (`.tpr`), so first `grompp` has to pack the system:

<div class="alert alert-block alert-warning">
<b>What, which, how.</b> Every GROMACS run starts from three files:
<ol>
<li>coordinates, <code>.gro</code>: <b>what</b> system;</li>
<li>topology, <code>.top</code>: <b>which</b> model;</li>
<li>parameters, <code>.mdp</code>: <b>how</b> to simulate.</li>
</ol>
<code>grompp</code> combines them into one <code>.tpr</code> file. <code>genion</code> needs a <code>.tpr</code> only to know the charges, so any valid <code>.mdp</code> will do: we use the minimisation one.
</div>

In [ ]:
%%bash
cd cln025_model1/00-build
gmx grompp -f ../em.mdp -c solvated.gro -p topol.top -o ions.tpr > grompp_ions.out 2>&1 || grep -A 5 "^WARNING" grompp_ions.out

<div class="alert alert-block alert-warning"><b>It failed.</b> Raise your hand, or read the output? The warning says that the system has a net charge, which with Ewald (PME) electrostatics causes artefacts. That is true, and it is exactly what we are about to fix with the ions. This is the one case in the course where we ignore a warning, with <code>-maxwarn 1</code>: never use <code>-maxwarn</code> to silence a warning you have not read and understood.</div>

In [ ]:
%%bash
cd cln025_model1/00-build
gmx grompp -f ../em.mdp -c solvated.gro -p topol.top -o ions.tpr -maxwarn 1 > grompp_ions.out 2>&1 || tail -n 20 grompp_ions.out
echo SOL | gmx genion -s ions.tpr -o system.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15 > genion.out 2>&1 || tail -n 20 genion.out
tail -n 4 topol.top
echo "atoms: $(sed -n 2p system.gro)"

`genion` asks which group of molecules the ions may replace; `echo SOL |` answers "the water". `-neutral` adds the counter-ions that cancel the peptide's charge (two extra Na⁺), and `-conc 0.15` adds Na⁺/Cl⁻ pairs for 0.15 mol/L. It updates the topology with the ions and the water they replaced.

The result, 3,630 atoms, is the system of `../common/structures/cln025_solvated`, which you used for the LAB 0 benchmark. Like `solvate`, run `genion` only once: each run takes the same `ions.tpr` but updates the topology again, and the two stop matching.

Look at the whole system in VMD (`vmd cln025_model1/00-build/system.gro`): show the water as Lines, the ions as VDW spheres and the peptide as NewCartoon, and `pbc box` in the Tk Console.

# 3. Energy minimisation

Water and ions were placed without regard for the forces between them, so some atoms sit far too close. Starting dynamics from there would produce huge forces and could blow the system apart. **Energy minimisation** first moves the atoms downhill in energy until the largest force is small.

The parameter file:

In [ ]:
!cat cln025_model1/em.mdp

| Setting | Meaning |
|---|---|
| `integrator = steep` | Steepest descent: move every atom along its force, with a step that grows while the energy goes down and shrinks when it goes up. Not really an integrator: there is no time |
| `emtol = 100.0` | **Stop** when the largest force on any atom is below 100 kJ mol⁻¹ nm⁻¹... |
| `nsteps = 50000` | ...**or** after 50,000 steps, whichever comes first |
| `emstep = 0.01` | Initial step size (nm) |
| `coulombtype = PME`, `rcoulomb = 1.0` | Electrostatics: computed directly up to 1.0 nm, and the long-range rest with the particle-mesh Ewald method |
| `vdwtype = Cut-off`, `rvdw = 1.0`, `DispCorr = EnerPres` | Lennard-Jones interactions up to 1.0 nm, plus an analytical correction for the energy and pressure beyond it |

Every stage of the course uses the same interaction settings; only the stage-specific ones change.

In [ ]:
%%bash
cd cln025_model1/01-em
gmx grompp -f ../em.mdp -c ../00-build/system.gro -p ../00-build/topol.top -o em.tpr > grompp.out 2>&1 || tail -n 20 grompp.out
gmx mdrun -deffnm em -ntmpi 1 > mdrun.out 2>&1 || tail -n 20 mdrun.out
grep -A 3 "converged" em.log
ls

- `-deffnm em` ("default file names") reads `em.tpr` and names every output `em.*`: `em.gro` (final coordinates), `em.edr` (energies), `em.log` (the log), `em.trr` (a trajectory, unused here).
- `-ntmpi 1` runs one process that uses all your CPUs through threads: for a system of a few thousand atoms this is the fastest choice, and the one used for the LAB 0 benchmark.

Check that it worked: the minimisation must have **converged** to a maximum force below `emtol`. Then plot the potential energy, which `gmx energy` extracts from the `.edr` file:

In [ ]:
%%bash
cd cln025_model1/01-em
echo Potential | gmx energy -f em.edr -o potential.xvg > energy.out 2>&1 || tail -n 20 energy.out

In [ ]:
data = np.loadtxt("cln025_model1/01-em/potential.xvg", comments=["#", "@"])
step, energy = data[:, 0], data[:, 1]

plt.plot(step, energy)
plt.xlabel("minimisation step")
plt.ylabel("potential energy (kJ/mol)")
plt.title("Energy minimisation")
plt.tight_layout()
plt.savefig("cln025_model1/01-em/potential.png", dpi=300)

The energy should fall steeply, then flatten out. Some steps look flat on this scale but still lower the largest force; that is what `emtol` checks.

# 4. Equilibration

The minimised system has no temperature: nothing moves yet. Equilibration brings it to the conditions of the production run in two stages, while the peptide is held near its starting structure so that the water can arrange itself around it:

1. **NVT**: constant volume, bringing the temperature to 300 K;
2. **NPT**: constant pressure, letting the box adjust to 1 bar, and the density with it.

## 4.1 Position restraints

`pdb2gmx` wrote `posre.itp`: a list of the peptide's heavy atoms (everything except hydrogen), each tied to a reference position $R_i$ by a harmonic spring:

$$V_{pr}(r_i) = \frac{1}{2} k_{pr} \, |r_i - R_i|^2$$

with $k_{pr}$ = 1000 kJ mol⁻¹ nm⁻² in each direction. The atoms can still move, but not far.

In [ ]:
!head -n 12 cln025_model1/00-build/posre.itp
!grep -B 1 -A 2 "POSRES" cln025_model1/00-build/topol.top

The restraints are in the topology only `#ifdef POSRES`: the line `define = -DPOSRES` of the equilibration `.mdp` files switches them on, and the production file leaves them off. The reference positions $R_i$ come from the file given to `grompp` with `-r`.

## 4.2 NVT: bringing the system to 300 K

In [ ]:
!cat cln025_model1/nvt.mdp

| Setting | Meaning |
|---|---|
| `integrator = md`, `dt = 0.002` | Leap-frog integration of Newton's equations with a 2 fs time step |
| `nsteps = 50000` | 50,000 × 2 fs = 100 ps |
| `constraints = h-bonds` | Bonds to hydrogen vibrate too fast for a 2 fs step (their period is about 10 fs), so their length is fixed by the LINCS algorithm |
| `tcoupl = v-rescale`, `tau-t = 0.1`, `ref-t = 300` | The thermostat: rescales the velocities towards 300 K with a time constant of 0.1 ps, plus a random term, so that the system samples the canonical ensemble (Bussi et al., *J Chem Phys* 2007, [doi:10.1063/1.2408420](https://doi.org/10.1063/1.2408420)) |
| `tc-grps = System` | One thermostat for the whole system: a few thousand atoms are too few to split into separately coupled groups |
| `pcoupl = no` | No barostat: constant volume |
| `gen-vel = yes`, `gen-temp = 300`, `gen-seed = -1` | Draw initial velocities at 300 K with a new random seed each time, so that every run is a different replica |
| `nstxout-compressed`, `nstenergy`, `nstlog` | How often coordinates (`.xtc`), energies (`.edr`) and log lines are written |

How long is a run? Total time = time step × number of steps:

In [ ]:
dt = 0.002        # ps
nsteps = 50000
print(f"{dt * nsteps:.0f} ps")

<div class="alert alert-block alert-danger"><b>Nerd corner: initial velocities.</b> With <code>gen-vel = yes</code> GROMACS gives each atom a velocity drawn from the Maxwell–Boltzmann distribution at <code>gen-temp</code>,

$$p(v_i) = \sqrt{\frac{m_i}{2\pi k_B T}} \exp\left(-\frac{m_i v_i^2}{2 k_B T}\right)$$

It then removes the motion of the centre of mass and scales all velocities so that the kinetic energy matches $T$ exactly.
</div>

In [ ]:
%%bash
cd cln025_model1/02-nvt
gmx grompp -f ../nvt.mdp -c ../01-em/em.gro -r ../01-em/em.gro -p ../00-build/topol.top -o nvt.tpr > grompp.out 2>&1 || tail -n 20 grompp.out
gmx mdrun -deffnm nvt -ntmpi 1 > mdrun.out 2>&1 || tail -n 20 mdrun.out
grep -B 1 "Performance:" nvt.log
echo Temperature | gmx energy -f nvt.edr -o temperature.xvg > energy.out 2>&1 || tail -n 20 energy.out

`Performance` is your speed on this run, in ns/day, as in LAB 0. Now the temperature:

In [ ]:
data = np.loadtxt("cln025_model1/02-nvt/temperature.xvg", comments=["#", "@"])
time, temperature = data[:, 0], data[:, 1]

plt.plot(time, temperature, lw=0.8)
plt.axhline(300, color="black", ls="--")
plt.xlabel("time (ps)")
plt.ylabel("temperature (K)")
plt.title("NVT equilibration")
plt.tight_layout()
print(f"mean {temperature[len(temperature) // 2:].mean():.1f} K, "
      f"standard deviation {temperature[len(temperature) // 2:].std():.1f} K (second half)")

<div class="alert alert-block alert-info">The temperature fluctuates by a few kelvin around 300 K, and it should. The temperature is computed from the kinetic energy of a few thousand atoms: why must it fluctuate, and why would the fluctuations be smaller in a larger system?</div>

## 4.3 NPT: bringing the system to 1 bar

What changes from the NVT file:

In [ ]:
!diff cln025_model1/nvt.mdp cln025_model1/npt.mdp

| Setting | Meaning |
|---|---|
| `nsteps = 250000` | 500 ps |
| `pcoupl = C-rescale`, `tau-p = 2.0`, `ref-p = 1.0` | The barostat: rescales the box and the coordinates towards 1 bar with a time constant of 2 ps. C-rescale samples the correct NPT ensemble (Bernetti & Bussi, *J Chem Phys* 2020, [doi:10.1063/5.0020514](https://doi.org/10.1063/5.0020514)); the Berendsen barostat of older versions of this course does not |
| `compressibility = 4.5e-5` | Compressibility of water (bar⁻¹): how much the box changes for a given pressure difference |
| `refcoord-scaling = com` | Move the restraint reference positions with the box when it is rescaled |
| `continuation = yes`, `gen-vel = no` | Continue the NVT run instead of starting over... |

...and `grompp -t nvt.cpt` passes the NVT **checkpoint**, with its velocities and thermostat state:

In [ ]:
%%bash
cd cln025_model1/03-npt
gmx grompp -f ../npt.mdp -c ../02-nvt/nvt.gro -r ../02-nvt/nvt.gro -t ../02-nvt/nvt.cpt -p ../00-build/topol.top -o npt.tpr > grompp.out 2>&1 || tail -n 20 grompp.out
gmx mdrun -deffnm npt -ntmpi 1 > mdrun.out 2>&1 || tail -n 20 mdrun.out
grep -B 1 "Performance:" npt.log
printf "Pressure\nDensity\n" | gmx energy -f npt.edr -o pressure_density.xvg > energy.out 2>&1 || tail -n 20 energy.out

In [ ]:
data = np.loadtxt("cln025_model1/03-npt/pressure_density.xvg", comments=["#", "@"])
time, pressure, density = data[:, 0], data[:, 1], data[:, 2]

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(7, 5))
axes[0].plot(time, pressure, lw=0.5)
axes[0].axhline(1, color="black", ls="--")
axes[0].set_ylabel("pressure (bar)")
axes[1].plot(time, density, lw=0.8)
axes[1].axhline(1000, color="black", ls="--")
axes[1].set_ylabel("density (kg/m$^3$)")
axes[1].set_xlabel("time (ps)")
fig.suptitle("NPT equilibration")
fig.tight_layout()
fig.savefig("cln025_model1/03-npt/pressure_density.png", dpi=300)

half = len(time) // 2
print(f"second half: pressure {pressure[half:].mean():.0f} ± {pressure[half:].std():.0f} bar, "
      f"density {density[half:].mean():.1f} ± {density[half:].std():.1f} kg/m3")

<div class="alert alert-block alert-info">The pressure swings by hundreds of bar from one frame to the next, while the density hardly changes. Why is that not a contradiction? (Think of how much the volume of water changes for a pressure change of 100 bar, using the compressibility above.)</div>

The system is now at 300 K and 1 bar, with the water arranged around the peptide: ready for production.

# 5. Production at three temperatures

The production runs are where the data come from. You will run three, from the same equilibrated state, that differ **only in the temperature of the thermostat**:

| Run | Temperature | Why |
|---|---|---|
| `md_10K` | 10 K | A negative control: this is what "no dynamics" looks like |
| `md_300K` | 300 K | Room temperature: does the hairpin stay folded? |
| `md_600K` | 600 K | Far above the melting temperature: the hairpin unfolds |

## 5.1 The production parameters

What changes from the NPT file:

In [ ]:
!diff cln025_model1/npt.mdp cln025_model1/md.mdp

- No `define = -DPOSRES`: the restraints are off, the peptide is free.
- The **barostat stays on**: the production runs sample the NPT ensemble, at 1 bar, like an experiment in a test tube. (Older versions of this course switched the barostat off in production without saying so.)
- Coordinates are saved every 5000 steps (10 ps) to the `.xtc` trajectory; that is what LAB 4 analyses.
- `nsteps` sets the length. You choose it now.

## 5.2 How long?

Your runs have to fit in the time you have. Enter your ns/day from LAB 0 and the length of each run:

In [ ]:
ns_per_day = None   # replace None with the ns/day you measured in LAB 0
length_ns = 10      # length of each production run, in ns

n_runs = 4          # 10 K, 300 K and 600 K, plus the GROMOS run of section 6
if ns_per_day is None:
    print("Set ns_per_day to the value you measured in LAB 0, then run this cell again.")
else:
    hours = length_ns / ns_per_day * 24
    print(f"One {length_ns} ns run takes about {hours:.1f} hours; all {n_runs} runs about {n_runs * hours:.1f} hours.")

Choose `length_ns` so that the four runs fit, one after the other, in the time you can leave the VM running (a night, for example), with a margin: the VM is slower while you use the computer for other things. Your NVT and NPT runs above printed their own speed, which may differ a little from the LAB 0 benchmark.

Ten nanoseconds are enough to see what the three temperatures do. They are far too short to see CLN025 unfold and fold again at 300 K: the folding and unfolding of CLN025 take hundreds of nanoseconds or more (its measured relaxation times are about 100 ns; Davis et al., *J Am Chem Soc* 2012, [doi:10.1021/ja3046734](https://doi.org/10.1021/ja3046734)). LAB 4 therefore also provides longer reference runs.

## 5.3 Three parameter files

The next cell writes one parameter file per temperature, changing only `nsteps`, `ref-t` and, for 600 K, the barostat:

In [ ]:
import re

nsteps = round(length_ns * 1000 / 0.002)
base = open("cln025_model1/md.mdp").read()
for T in (10, 300, 600):
    mdp = re.sub(r"(?m)^nsteps .*", f"nsteps          = {nsteps}", base)
    mdp = re.sub(r"(?m)^ref-t .*", f"ref-t           = {T}", mdp)
    if T == 600:
        mdp = re.sub(r"(?m)^pcoupl .*", "pcoupl          = no        ; constant volume at 600 K", mdp)
    with open(f"cln025_model1/04-md/md_{T}K.mdp", "w") as f:
        f.write(mdp)
    print(f"md_{T}K.mdp: {nsteps} steps = {length_ns} ns at {T} K")

In [ ]:
!diff cln025_model1/04-md/md_300K.mdp cln025_model1/04-md/md_10K.mdp
!diff cln025_model1/04-md/md_300K.mdp cln025_model1/04-md/md_600K.mdp

## 5.4 Why 600 K at constant volume

At 1 bar, water boils at 373 K. With the barostat on at 600 K, the box would expand without limit until the water became a dilute gas and the simulation became meaningless (or crashed). At **constant volume** the water keeps the density of the liquid: it becomes a very hot, compressed liquid under a pressure of thousands of bar (you will see it in section 8). That is nothing like physiological conditions: it is simply a fast way to unfold the hairpin. The 10 K run keeps the barostat: the water freezes into a glass and hardly anything moves.

## 5.5 The run input files

All three runs start from the end of the NPT equilibration, with its velocities: the thermostat brings each one to its own temperature within a few picoseconds.

In [ ]:
%%bash
cd cln025_model1/04-md
for T in 10K 300K 600K; do
    gmx grompp -f md_$T.mdp -c ../03-npt/npt.gro -t ../03-npt/npt.cpt -p ../00-build/topol.top -o md_$T.tpr > grompp_$T.out 2>&1 || tail -n 20 grompp_$T.out
done
ls *.tpr

The runs start in section 7, together with the run of the next section.

# 6. A second force field: GROMOS 54A7

Everything a simulation predicts depends on the force field. For CLN025 the difference is large: in a comparison of four force fields, AMBER ff14SB (a close relative of the course's amber99sb-ildn) stabilised the folded hairpin too much, while GROMOS 54A7 agreed well with the experimental populations of folded and unfolded states and reproduced the experimental folding times best (Kamenik et al., *J Chem Phys* 2020, [doi:10.1063/5.0022135](https://doi.org/10.1063/5.0022135)).

You will therefore set up the same capped molecule a second time, with GROMOS 54A7, and compare the two in LAB 4. Both force fields come with GROMACS.

| | Main | Comparison |
|---|---|---|
| Force field | amber99sb-ildn | gromos54a7 |
| Hydrogens | Every hydrogen is an atom | **United atom**: the hydrogens of CH, CH₂ and CH₃ groups are merged into their carbon |
| Water | TIP3P | SPC |
| Electrostatics | PME, 1.0 nm cut-off | Reaction field beyond a 1.4 nm cut-off, with the relative permittivity of the surrounding medium set to 61 |
| Lennard-Jones | 1.0 nm cut-off, dispersion correction | 1.4 nm cut-off, no correction |
| Parameter files | `../common/mdp/*.mdp` | `../common/mdp/gromos54a7/*.mdp` |

**A force field comes with the simulation settings it was parametrised with.** GROMOS was parametrised with a reaction field and a 1.4 nm cut-off (Diem & Oostenbrink, *J Chem Theory Comput* 2020, [doi:10.1021/acs.jctc.0c00509](https://doi.org/10.1021/acs.jctc.0c00509)), and a single 1.4 nm cut-off as used by current GROMACS reproduces its behaviour (Silva et al., *J Chem Theory Comput* 2018, [doi:10.1021/acs.jctc.8b00758](https://doi.org/10.1021/acs.jctc.8b00758)). Running GROMOS parameters with AMBER's settings would simulate a different model.

Note that the comparison changes the force field and the water model together: each force field was parametrised with its own water. GROMOS was also parametrised with every bond length constrained; the course constrains only bonds to hydrogen with both force fields, so that the time step and the rest of the protocol stay the same.

In [ ]:
!diff ../common/mdp/md.mdp ../common/mdp/gromos54a7/md.mdp

## 6.1 Topology

The folder is built by hand this time. `build_caps.py --gromos` names the acetyl carbon as GROMOS expects:

In [ ]:
%%bash
mkdir -p cln025_model1_gromos/{00-build,01-em,02-nvt,03-npt,04-md}
cp ../common/mdp/gromos54a7/*.mdp cln025_model1_gromos/
python3 ../common/scripts/build_caps.py ../common/structures/2RVD.pdb 1 cln025_model1_gromos/00-build/cln025_capped.pdb --gromos > /dev/null
cd cln025_model1_gromos/00-build
printf "2\n2\n" | gmx pdb2gmx -f cln025_capped.pdb -o cln025.gro -p topol.top -i posre.itp \
                               -ff gromos54a7 -water spc -ignh -ter > pdb2gmx.out 2>&1 || tail -n 20 pdb2gmx.out
grep -E -A 4 "^Select (start|end) terminus" pdb2gmx.out
grep -E "Now there are|Total charge" pdb2gmx.out

GROMOS attaches the chain ends to the first and last residue, and would treat ACE and NH2 as charged ends. `-ter` makes `pdb2gmx` ask for each end instead, and `printf "2\n2\n"` answers **None** twice: the caps already are the ends of the chain.

<div class="alert alert-block alert-info">The same peptide, the same charge, but how many atoms? Compare with the AMBER topology of section 2.2, and explain the difference.</div>

## 6.2 Box, water and ions

The same commands as for AMBER, with SPC water. Look at what `grompp` says this time:

In [ ]:
%%bash
cd cln025_model1_gromos/00-build
gmx editconf -f cln025.gro -o box.gro -c -d 1.0 -bt dodecahedron > editconf.out 2>&1 || tail -n 20 editconf.out
gmx solvate -cp box.gro -cs spc216.gro -o solvated.gro -p topol.top > solvate.out 2>&1 || tail -n 20 solvate.out
gmx grompp -f ../em.mdp -c solvated.gro -p topol.top -o ions.tpr > grompp_ions.out 2>&1 || grep -E -A 9 "^(WARNING|NOTE)" grompp_ions.out

Two messages. The net charge is now only a NOTE: without Ewald electrostatics it causes no artefacts, and the ions will remove it anyway. The WARNING is new: GROMACS warns about **every** GROMOS system that the force field was parametrised with a twin-range cut-off scheme that current GROMACS no longer supports. The authors of GROMOS tested exactly this and found the effect minor (Diem & Oostenbrink 2020), and Silva et al. (2018) found the single cut-off consistent with the force field. Having read it and checked the literature, we accept this one warning with `-maxwarn 1` in every GROMOS `grompp`.

In [ ]:
%%bash
cd cln025_model1_gromos/00-build
gmx grompp -f ../em.mdp -c solvated.gro -p topol.top -o ions.tpr -maxwarn 1 > grompp_ions.out 2>&1 || tail -n 20 grompp_ions.out
echo SOL | gmx genion -s ions.tpr -o system.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15 > genion.out 2>&1 || tail -n 20 genion.out
tail -n 4 topol.top
echo "atoms: $(sed -n 2p system.gro)   (AMBER: $(sed -n 2p ../../cln025_model1/00-build/system.gro))"

## 6.3 Minimisation and equilibration

The same three stages, with the GROMOS parameter files:

In [ ]:
%%bash
cd cln025_model1_gromos/01-em
gmx grompp -f ../em.mdp -c ../00-build/system.gro -p ../00-build/topol.top -o em.tpr -maxwarn 1 > grompp.out 2>&1 || tail -n 20 grompp.out
gmx mdrun -deffnm em -ntmpi 1 > mdrun.out 2>&1 || tail -n 20 mdrun.out
grep -A 3 "converged" em.log

cd ../02-nvt
gmx grompp -f ../nvt.mdp -c ../01-em/em.gro -r ../01-em/em.gro -p ../00-build/topol.top -o nvt.tpr -maxwarn 1 > grompp.out 2>&1 || tail -n 20 grompp.out
gmx mdrun -deffnm nvt -ntmpi 1 > mdrun.out 2>&1 || tail -n 20 mdrun.out
grep -B 1 "Performance:" nvt.log

cd ../03-npt
gmx grompp -f ../npt.mdp -c ../02-nvt/nvt.gro -r ../02-nvt/nvt.gro -t ../02-nvt/nvt.cpt -p ../00-build/topol.top -o npt.tpr -maxwarn 1 > grompp.out 2>&1 || tail -n 20 grompp.out
gmx mdrun -deffnm npt -ntmpi 1 > mdrun.out 2>&1 || tail -n 20 mdrun.out
grep -B 1 "Performance:" npt.log
printf "Temperature\nDensity\n" | gmx energy -f npt.edr > energy.out 2>&1 || tail -n 20 energy.out
grep -A 3 "^Energy" energy.out

Is the GROMOS system faster or slower than the AMBER one? It has fewer atoms, but a longer cut-off, so each atom interacts with more neighbours.

The production run at 300 K, with the same length as the AMBER runs:

In [ ]:
mdp = open("cln025_model1_gromos/md.mdp").read()
mdp = re.sub(r"(?m)^nsteps .*", f"nsteps          = {nsteps}", mdp)
with open("cln025_model1_gromos/04-md/md_300K.mdp", "w") as f:
    f.write(mdp)

In [ ]:
%%bash
cd cln025_model1_gromos/04-md
gmx grompp -f md_300K.mdp -c ../03-npt/npt.gro -t ../03-npt/npt.cpt -p ../00-build/topol.top -o md_300K.tpr -maxwarn 1 > grompp_300K.out 2>&1 || tail -n 20 grompp_300K.out
ls *.tpr

# 7. Running the simulations

Four runs are ready. The next cell writes a script that runs them one after the other; each run uses all the CPUs of the VM.

In [ ]:
%%bash
cat > run_production.sh << 'EOF'
#!/bin/bash
# LAB 3 production runs, one after the other. Start from a terminal with:
#   nohup bash run_production.sh > run_production.out 2>&1 &
cd "$(dirname "$0")"
for T in 300K 600K; do
    echo "$(date '+%F %T')  start AMBER $T"
    (cd cln025_model1/04-md && gmx mdrun -deffnm md_$T -ntmpi 1 -v > mdrun_$T.out 2>&1)
done
echo "$(date '+%F %T')  start GROMOS 300K"
(cd cln025_model1_gromos/04-md && gmx mdrun -deffnm md_300K -ntmpi 1 -v > mdrun_300K.out 2>&1)
echo "$(date '+%F %T')  start AMBER 10K"
(cd cln025_model1/04-md && gmx mdrun -deffnm md_10K -ntmpi 1 -v > mdrun_10K.out 2>&1)
echo "$(date '+%F %T')  all done"
EOF
cat run_production.sh

Start it **from a terminal** (JupyterLab: File → New → Terminal, or the Course Terminal icon), in the background with `nohup` as in LAB 1, so that it keeps running when you close the notebook:

```bash
cd ~/molbiomech/labs/03-ClassicalMD
nohup bash run_production.sh > run_production.out 2>&1 &
```

- Don't start other simulations meanwhile: they would share the CPUs and both would slow down.
- You can close JupyterLab. Don't shut the VM down: that stops the run (see 7.2 to resume it).
- `top` in a terminal shows `gmx` using the CPUs.

## 7.1 Checking progress

With `-v`, `mdrun` keeps printing when it expects to finish. This cell shows the log of the script and the last estimate of each run; run it whenever you like:

In [ ]:
%%bash
cat run_production.out 2>/dev/null || echo "run_production.sh has not been started yet"
for out in cln025_model1/04-md/mdrun_*.out cln025_model1_gromos/04-md/mdrun_*.out; do
    [ -f "$out" ] && echo "$out: $(tr '\r' '\n' < "$out" | grep -E "will finish|Finished mdrun|^Performance" | tail -n 1)"
done

## 7.2 If a run stops, or is too short

`mdrun` writes a **checkpoint** (`.cpt`) every 15 minutes. If a run stops (the VM was shut down, the computer went to sleep), continue it from where it stopped, in its folder:

```bash
gmx mdrun -deffnm md_300K -cpi md_300K.cpt -ntmpi 1 -v
```

GROMACS appends the new frames and energies to the existing files. The same trick extends a finished run: `convert-tpr` adds time (in ps) to the run input file, and `mdrun -cpi` continues:

```bash
gmx convert-tpr -s md_300K.tpr -extend 10000 -o md_300K.tpr    # 10 ns more
gmx mdrun -deffnm md_300K -cpi md_300K.cpt -ntmpi 1 -v
```

## 7.3 If your runs could not finish

The course's own runs of this lab, made with exactly these commands, are a reference dataset. Download them in a terminal:

```bash
course-fetch lab03-production
```

They land in `reference/`, laid out like your own folders (`reference/cln025_model1/04-md/...`, `reference/cln025_model1_gromos/04-md/...`). Section 8 and LAB 4 work on either. Choose here which runs the next cells read: `.` for your own, `reference` for the course's.

In [ ]:
%env RUNS=.

# 8. A first look at the results

Run this section when the four runs have finished. First, the speed each run achieved, to compare with LAB 0 and to write in your run log:

In [ ]:
%%bash
cd $RUNS
grep -H "Performance:" cln025_model1/04-md/md_*.log cln025_model1_gromos/04-md/md_*.log

## 8.1 Did each run do what we asked?

In [ ]:
%%bash
cd $RUNS
for run in cln025_model1/04-md/md_10K cln025_model1/04-md/md_300K cln025_model1/04-md/md_600K cln025_model1_gromos/04-md/md_300K; do
    printf "Temperature\nPressure\n" | gmx energy -f $run.edr -o ${run}_TP.xvg > ${run}_energy.out 2>&1 || tail -n 20 ${run}_energy.out
done

In [ ]:
import os

runs_dir = os.environ["RUNS"]
runs = {"AMBER 10 K": "cln025_model1/04-md/md_10K", "AMBER 300 K": "cln025_model1/04-md/md_300K",
        "AMBER 600 K": "cln025_model1/04-md/md_600K", "GROMOS 300 K": "cln025_model1_gromos/04-md/md_300K"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
print(f"{'run':<13} {'temperature (K)':>16} {'pressure (bar)':>16}   (second half of each run)")
for label, run in runs.items():
    data = np.loadtxt(f"{runs_dir}/{run}_TP.xvg", comments=["#", "@"])
    time, temperature, pressure = data[:, 0] / 1000, data[:, 1], data[:, 2]
    axes[0].plot(time, temperature, lw=0.8, label=label)
    axes[1].plot(time, pressure, lw=0.5, label=label)
    half = len(time) // 2
    print(f"{label:<13} {temperature[half:].mean():9.1f} ± {temperature[half:].std():4.1f} "
          f"{pressure[half:].mean():9.0f} ± {pressure[half:].std():4.0f}")
axes[0].set_ylabel("temperature (K)")
axes[1].set_ylabel("pressure (bar)")
for ax in axes:
    ax.set_xlabel("time (ns)")
axes[0].legend()
fig.tight_layout()

<div class="alert alert-block alert-info">Did each thermostat reach its temperature? What happened to the pressure at 600 K, and why (section 5.4)? The three other runs have the barostat on: is their average pressure close to 1 bar, given how much it fluctuates?</div>

## 8.2 Looking at a trajectory

Load a raw trajectory into VMD and parts of the peptide seem to jump across the box, with bonds stretched from one side to the other. Nothing is wrong: with periodic boundary conditions an atom that leaves one face re-enters through the opposite one, and the program draws the bonds of the broken molecule across the box. `gmx trjconv` makes the molecules whole again (`-pbc mol`) and puts the peptide at the centre (`-center`). Here it also keeps only the peptide, which makes the files much smaller:

In [ ]:
%%bash
cd $RUNS/cln025_model1/04-md
for T in 10K 300K 600K; do
    printf "Protein\nProtein\n" | gmx trjconv -s md_$T.tpr -f md_$T.xtc -o md_${T}_protein.xtc -pbc mol -center > trjconv_$T.out 2>&1 || tail -n 20 trjconv_$T.out
done
printf "Protein\nProtein\n" | gmx trjconv -s md_300K.tpr -f md_300K.xtc -o protein.gro -pbc mol -center -dump 0 > trjconv_gro.out 2>&1 || tail -n 20 trjconv_gro.out
ls -lh *.xtc

`trjconv` asks two questions: the group to centre, and the group to write; `printf` answers `Protein` to both. Watch the three runs in VMD, from a terminal in `cln025_model1/04-md`:

```bash
vmd protein.gro md_10K_protein.xtc
vmd protein.gro md_300K_protein.xtc
vmd protein.gro md_600K_protein.xtc
```

Use NewCartoon coloured by Structure, and Tyr2 and Trp9 as Licorice, as in LAB 2. With **Extensions → Analysis → RMSD Trajectory Tool** you can align the frames on the backbone, so that the peptide stops tumbling.

## 8.3 The end-to-end distance

The distance between the Cα atoms of Tyr1 and Tyr10 is small (about 0.5 nm) when the hairpin is closed, and grows when it opens. It is the coordinate used again in LAB 4, for a free-energy profile, and in LAB 6, where the peptide is pulled along it.

In [ ]:
%%bash
cd $RUNS
for run in cln025_model1/04-md/md_10K cln025_model1/04-md/md_300K cln025_model1/04-md/md_600K cln025_model1_gromos/04-md/md_300K; do
    gmx distance -s $run.tpr -f $run.xtc -select "resid 1 and name CA plus resid 10 and name CA" -oall ${run}_e2e.xvg > ${run}_distance.out 2>&1 || tail -n 20 ${run}_distance.out
done

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, run in runs.items():
    data = np.loadtxt(f"{runs_dir}/{run}_e2e.xvg", comments=["#", "@"])
    ax.plot(data[:, 0] / 1000, data[:, 1], lw=0.8, label=label)
ax.set_xlabel("time (ns)")
ax.set_ylabel("Tyr1 CA - Tyr10 CA distance (nm)")
ax.legend()
fig.tight_layout()
fig.savefig("e2e.png", dpi=300)

<div class="alert alert-block alert-info">Which runs keep the hairpin closed? When does the 600 K hairpin open, and does it ever close again? Do the AMBER and GROMOS runs at 300 K look different? A single short run cannot answer the last question: LAB 4 does it properly, with many longer runs.</div>

# 9. Exercises

## 9.1 Complete setup.sh

In LAB 1 your `setup.sh` created the folder of one NMR model and capped it. Add the commands of sections 2 to 5, so that

```bash
bash setup.sh 7 10
```

builds model 7 in water, minimises and equilibrates it, and runs 10 ns of production at 300 K. The second argument, the length in ns, is optional (default 10).

<div class="alert alert-block alert-warning">
<b>HINTS</b><br>
<code>set -e</code> stops the script at the first command that fails. <code>cd</code> into each stage folder and use relative paths as in this notebook. <code>${2:-10}</code> is the second argument, or 10 if there is none. <code>sed "s/^nsteps .*/nsteps = $nsteps/"</code> sets the length.
</div>

A solution is in `solution_setup.sh`.

**The class dataset**: your instructor gives each of you a different NMR model of 2RVD. When your four runs above are done, run your model with your script, in the background:

```bash
nohup bash setup.sh <your model> > setup.out 2>&1 &
```

Together, the class's runs start from all 20 models of the NMR ensemble: far more sampling than any one of you could collect, analysed together in LAB 4.

## 9.2 Wild-type chignolin

CLN025 was designed from **chignolin**, GYDPETGTWG (PDB [1UAO](https://www.rcsb.org/structure/1UAO), Honda et al., *Structure* 2004, [doi:10.1016/j.str.2004.05.022](https://doi.org/10.1016/j.str.2004.05.022)), by replacing the two terminal glycines with tyrosines (Hatfield et al., *J Phys Chem B* 2010, [doi:10.1021/jp910465e](https://doi.org/10.1021/jp910465e)). A copy is in `../common/structures/1UAO.pdb`. In a new folder `chignolin_wt`:

1. How many models does 1UAO contain? Which residues differ from 2RVD?
2. Build its topology with amber99sb-ildn and TIP3P, **without caps**: the chain keeps its charged ends (NH₃⁺ and COO⁻). What is the total charge, and where does it come from?
3. Build a rhombic dodecahedron with d = 1.0 nm, solvate it and add 0.15 M NaCl. How many atoms and ions does it have, compared with capped CLN025?

`pdb2gmx` reads only the first model of a multi-model file.

**Solution**

In [ ]:
%%bash
mkdir -p chignolin_wt && cd chignolin_wt
cp ../../common/structures/1UAO.pdb .
echo "models: $(grep -c '^MODEL' 1UAO.pdb)"
grep "^SEQRES" 1UAO.pdb
gmx pdb2gmx -f 1UAO.pdb -o wt.gro -p topol.top -i posre.itp -ff amber99sb-ildn -water tip3p -ignh > pdb2gmx.out 2>&1 || tail -n 20 pdb2gmx.out
grep -E "terminus|Now there are|Total charge" pdb2gmx.out
gmx editconf -f wt.gro -o box.gro -c -d 1.0 -bt dodecahedron > editconf.out 2>&1 || tail -n 20 editconf.out
gmx solvate -cp box.gro -cs spc216.gro -o solvated.gro -p topol.top > solvate.out 2>&1 || tail -n 20 solvate.out
gmx grompp -f ../cln025_model1/em.mdp -c solvated.gro -p topol.top -o ions.tpr -maxwarn 1 > grompp_ions.out 2>&1 || tail -n 20 grompp_ions.out
echo SOL | gmx genion -s ions.tpr -o system.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15 > genion.out 2>&1 || tail -n 20 genion.out
tail -n 4 topol.top
echo "atoms: $(sed -n 2p system.gro)   (capped CLN025: $(sed -n 2p ../cln025_model1/00-build/system.gro))"

Wild-type chignolin has Gly instead of Tyr at both ends, and free, charged ends: +1 on the N-terminus and −1 on the C-terminus, which cancel, plus −1 each from Asp3 and Glu5. Its total charge is −2, like capped CLN025, but for a different reason. The two charged ends attract each other; in a pulling experiment that attraction distorts the force (Zhao & Cheng, *Amino Acids* 2012, [doi:10.1007/s00726-011-1150-5](https://doi.org/10.1007/s00726-011-1150-5)), which is why the course caps CLN025.

# 10. Your deliverable

1. **A run log**: your NMR model; the box type and `d` you would choose, with your reasons (section 2.4); the number of atoms of the AMBER and GROMOS systems; the ns/day of each run from its `.log` file, compared with LAB 0; the wall-clock time; anything that went wrong and how you fixed it.
2. **The plots of section 8**, each with two or three sentences: what does each temperature do to the hairpin?
3. **Your run for the class dataset** (exercise 9.1), with its folder left in place for LAB 4.

As for the other simulation labs, present them in a few slides during a 15-minute group discussion.